### Basic Setup

In [65]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns
import config
from src.utils import (
    read_csv_file,
    reduce_mem_usage,
    save_csv_file
)
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from src.fe import *
import optuna
from tqdm.notebook import tqdm
from itertools import combinations
pd.set_option('display.max_columns', None)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [67]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [68]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)
classes = np.unique(y)

# Test data
test_data = test_raw.drop('id', axis=1)

In [69]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


### Blending

In [70]:
# Catgbm
oof_proba_catgbm = read_csv_file(r'artifacts\oof\Catgbm_seed42_v9_oof_proba.csv')
oof_proba_catgbm = oof_proba_catgbm.iloc[:, 1:]

test_proba_catgbm = read_csv_file(r'artifacts\test_proba\Catgbm_seed42_v9_test_proba.csv')
test_proba_catgbm = test_proba_catgbm.iloc[:, 1:]


# LR
oof_proba_lr = read_csv_file(r'artifacts\oof\Logistic_seed42_v21_oof_proba.csv')
oof_proba_lr = oof_proba_lr.iloc[:, 1:]

test_proba_lr = read_csv_file(r'artifacts\test_proba\Logistic_seed42_v21_test_proba.csv')
test_proba_lr = test_proba_lr.iloc[:, 1:]


# HistGBM
oof_proba_hist = read_csv_file(r'artifacts\oof\histgbm_seed42_v4_oof_proba.csv')
oof_proba_hist = oof_proba_hist.iloc[:, 1:]

test_proba_hist = read_csv_file(r'artifacts\test_proba\histgbm_seed42_v4_test_proba.csv')
test_proba_hist = test_proba_hist.iloc[:, 1:]


# ResNet
oof_proba_resnet = read_csv_file(r'artifacts\oof\resnet_seed42_v4_oof_proba.csv')
oof_proba_resnet = oof_proba_resnet.iloc[:, 1:]

test_proba_resnet = read_csv_file(r'artifacts\test_proba\resnet_seed42_v4_test_proba.csv')
test_proba_resnet = test_proba_resnet.iloc[:, 1:]


# XGBM
oof_proba_xgbm = read_csv_file(r'artifacts\oof\xgbm_seed42_v1_oof_proba.csv')
oof_proba_xgbm = oof_proba_xgbm.iloc[:, 1:]

test_proba_xgbm = read_csv_file(r'artifacts\test_proba\xgbm_seed42_v1_test_proba.csv')
test_proba_xgbm = test_proba_xgbm.iloc[:, 1:]


# RealMLP
oof_proba_realmlp = read_csv_file(r'artifacts\oof\realMLP_seed42_v5_oof_proba.csv')
oof_proba_realmlp = oof_proba_realmlp.iloc[:, 1:]

test_proba_realmlp = read_csv_file(r'artifacts\test_proba\realMLP_seed42_v5_test_proba.csv')
test_proba_realmlp = test_proba_realmlp.iloc[:, 1:]

Reading file from: artifacts\oof\Catgbm_seed42_v9_oof_proba.csv
Reading file from: artifacts\test_proba\Catgbm_seed42_v9_test_proba.csv
Reading file from: artifacts\oof\Logistic_seed42_v21_oof_proba.csv
Reading file from: artifacts\test_proba\Logistic_seed42_v21_test_proba.csv
Reading file from: artifacts\oof\histgbm_seed42_v4_oof_proba.csv
Reading file from: artifacts\test_proba\histgbm_seed42_v4_test_proba.csv
Reading file from: artifacts\oof\resnet_seed42_v4_oof_proba.csv
Reading file from: artifacts\test_proba\resnet_seed42_v4_test_proba.csv
Reading file from: artifacts\oof\xgbm_seed42_v1_oof_proba.csv
Reading file from: artifacts\test_proba\xgbm_seed42_v1_test_proba.csv
Reading file from: artifacts\oof\realMLP_seed42_v5_oof_proba.csv
Reading file from: artifacts\test_proba\realMLP_seed42_v5_test_proba.csv


In [71]:
oof_catgbm = oof_proba_catgbm.values
oof_lr = oof_proba_lr.values
oof_hist = oof_proba_hist.values
oof_resnet = oof_proba_resnet.values
oof_xgbm = oof_proba_xgbm.values
oof_realmlp = oof_proba_realmlp.values

test_catgbm = test_proba_catgbm.values
test_lr = test_proba_lr.values
test_hist = test_proba_hist.values
test_resnet = test_proba_resnet.values
test_xgbm = test_proba_xgbm.values
test_realmlp = test_proba_realmlp.values

# quick sanity check
print("CATGBM OOF shape:", oof_catgbm.shape)
print("LR OOF shape:", oof_lr.shape)
print("Hist OOF shape:", oof_hist.shape)
print("Resnet OOF shape:", oof_resnet.shape)
print("XGBM OOF shape:", oof_xgbm.shape)
print("RealMLP OOF shape:", oof_realmlp.shape)

CATGBM OOF shape: (630000, 3)
LR OOF shape: (630000, 3)
Hist OOF shape: (630000, 3)
Resnet OOF shape: (630000, 3)
XGBM OOF shape: (630000, 3)
RealMLP OOF shape: (630000, 3)


In [72]:
score_catgbm = balanced_accuracy_score(y, np.argmax(oof_catgbm, axis=1))
score_lr = balanced_accuracy_score(y, np.argmax(oof_lr, axis=1))
score_hist = balanced_accuracy_score(y, np.argmax(oof_hist, axis=1))
score_resnet = balanced_accuracy_score(y, np.argmax(oof_resnet, axis=1))
score_xgbm = balanced_accuracy_score(y, np.argmax(oof_xgbm, axis=1))
score_realmlp = balanced_accuracy_score(y, np.argmax(oof_realmlp, axis=1))

print(f"CATGBM alone: {score_catgbm:.6f}")
print(f"LR alone: {score_lr:.6f}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")

CATGBM alone: 0.971750
LR alone: 0.976472
HISTGBM alone: 0.972271
RESNET alone: 0.959926
XGBM alone: 0.978548
REALMLP alone: 0.978020


In [30]:
def objective(trial):

    w_catgbm = trial.suggest_float('w_catgbm', 0.0, 1.0)
    w_lr = trial.suggest_float('w_lr', 0.0, 1.0)
    w_hist = trial.suggest_float('w_hist', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    w_xgbm = trial.suggest_float('w_xgbm', 0.0, 1.0)
    w_realmlp = trial.suggest_float('w_realmlp', 0.0, 1.0)

    # normalize
    total = w_catgbm + w_lr + w_hist + w_resnet + w_xgbm + w_realmlp

    w_catgbm /= total
    w_lr /= total
    w_hist /= total
    w_resnet /= total
    w_xgbm /= total
    w_realmlp /= total

    # blend
    blend = (
        w_catgbm  * oof_catgbm +
        w_lr * oof_lr +
        w_hist * oof_hist +
        w_resnet * oof_resnet +
        w_xgbm * oof_xgbm +
        w_realmlp * oof_realmlp
    )

    preds = np.argmax(blend, axis=1)

    return balanced_accuracy_score(y, preds)


study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=3000, show_progress_bar=True)


# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best['w_catgbm'] +
    best['w_lr'] +
    best['w_hist'] +
    best['w_resnet'] +
    best['w_xgbm'] +
    best['w_realmlp']
)

w_catgbm = best['w_catgbm']  / total
w_lr = best['w_lr'] / total
w_hist = best['w_hist'] / total
w_resnet = best['w_resnet'] / total
w_xgbm = best['w_xgbm'] / total
w_realmlp = best['w_realmlp'] / total


# -------- PRINT --------
print(f"\n{'='*40}")
print(f"CATGBM alone: {score_catgbm:.6f}")
print(f"LR alone: {score_lr:.6f}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"RESNET alone: {score_resnet:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"REALMLP alone: {score_realmlp:.6f}")
print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - max(score_catgbm, score_lr, score_hist, score_resnet, score_xgbm, score_realmlp):.6f}")
print(f"{'='*40}")

print(f"CATGBM weight: {w_catgbm:.4f} ({w_catgbm:.1%})")
print(f"LR weight: {w_lr:.4f} ({w_lr:.1%})")
print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"RESNET weight: {w_resnet:.4f} ({w_resnet:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"REALMLP weight: {w_realmlp:.4f} ({w_realmlp:.1%})")

  0%|          | 0/3000 [00:00<?, ?it/s]


CATGBM alone:  0.971750
LR alone:      0.976472
HISTGBM alone: 0.972271
RESNET alone:  0.959926
XGBM alone:    0.978548
REALMLP alone: 0.974044
Best blend:    0.978995
Gain: +0.000447
CATGBM weight:  0.0010 (0.1%)
LR weight:      0.4416 (44.2%)
HIST weight:    0.0009 (0.1%)
RESNET weight:  0.0013 (0.1%)
XGBM weight:    0.5548 (55.5%)
REALMLP weight: 0.0005 (0.0%)


In [32]:
blend_oof  = w_catgbm * oof_catgbm + w_lr * oof_lr + w_hist * oof_hist + w_resnet * oof_resnet + w_xgbm * oof_xgbm + w_realmlp + oof_realmlp
blend_test = w_catgbm * test_catgbm + w_lr * test_lr + w_hist * test_hist + w_resnet * test_resnet + w_xgbm * test_xgbm + w_realmlp + test_realmlp

# -------- SAVE BLENDED FILES --------
blend_oof_df = pd.DataFrame(blend_oof, columns=[str(c) for c in classes])
blend_oof_df.insert(0, "id", train_ids)

save_csv_file(
    blend_oof_df,
    os.path.join(
        config.OOF_DIR,
        'blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_oof_proba.csv'
    )
)

blend_test_df = pd.DataFrame(blend_test, columns=[str(c) for c in classes])
blend_test_df.insert(0, "id", test_ids)

save_csv_file(
    blend_test_df,
    os.path.join(
        config.TEST_PROBA_DIR,
        'blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_test_proba.csv'
    )
)

Saving file to: artifacts\oof\blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_oof_proba.csv
Saving file to: artifacts\test_proba\blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_test_proba.csv


### Threshold Tuning

In [41]:
oof = read_csv_file(r'artifacts\oof\realMLP_seed42_v5_oof_proba.csv')
oof =oof.iloc[:, 1:]

test_proba = read_csv_file(r'artifacts\test_proba\realMLP_seed42_v5_test_proba.csv')
test_proba = test_proba.iloc[:, 1:]

Reading file from: artifacts\oof\realMLP_seed42_v5_oof_proba.csv
Reading file from: artifacts\test_proba\realMLP_seed42_v5_test_proba.csv


In [42]:
n_classes = 3

def predict_with_thresholds(probs, thresholds):
    scaled = probs / (np.array(thresholds))
    return np.argmax(scaled, axis=1)

def objective(trial):
    thresholds = [
        trial.suggest_float(f"thresh_{i}", 0.01, 0.99)
        for i in range(n_classes)
    ]
    preds = predict_with_thresholds(oof, thresholds)
    return balanced_accuracy_score(y, preds)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=5000, show_progress_bar=True)

best_thresholds = [study.best_params[f"thresh_{i}"] for i in range(n_classes)]

baseline  = balanced_accuracy_score(y, np.argmax(oof, axis=1))
optimised = balanced_accuracy_score(y, predict_with_thresholds(oof, best_thresholds))

abs_gain = optimised - baseline
pct_gain = (abs_gain / baseline) * 100 if baseline != 0 else 0

print(f"Baseline BACC: {baseline}")
print(f"Optimised BACC: {optimised}")
print(f"Gain: +{abs_gain:.5f} ({pct_gain:.5f}%)")
print(f"Thresholds : class_0={best_thresholds[0]:.5f} | class_1={best_thresholds[1]:.5f} | class_2={best_thresholds[2]:.5f}")

  0%|          | 0/5000 [00:00<?, ?it/s]

Baseline BACC: 0.9780202691943054
Optimised BACC: 0.9781828815330583
Gain: +0.00016 (0.01663%)
Thresholds : class_0=0.92388 | class_1=0.97091 | class_2=0.73058


In [35]:
out = predict_with_thresholds(test_proba, best_thresholds)
out = pd.DataFrame({'id': test_ids, 'Irrigation_Need': out})
out.Irrigation_Need = out.Irrigation_Need.map(config.TARGET_MAP_INV)

save_path = os.path.join(config.SUBMISSIONS_DIR, 'blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_thresh_tuned_submission.csv')
save_csv_file(out, save_path)

Saving file to: artifacts\submissions\blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_test_proba_thresh_tuned_submission.csv
